# Identified-Flow Dataset Audit

This notebook validates the separately processed CIC-IDS2017 dataset used for leakage-controlled experiments.

- Source: `data/raw_identified_flows/`
- Processed file: `data/processed/identified_flows.parquet`
- Preprocessing script: `src/prepare_identified_flows.py`

Original CSV files and the legacy `clean_flows.parquet` are preserved.


## 1. Load the processed dataset

In [7]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "processed" / "identified_flows.parquet").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "processed" / "identified_flows.parquet"
df = pd.read_parquet(DATA_PATH)

print("File:", DATA_PATH)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Labels:", df["Label"].nunique())
print("Missing values:", f"{int(df.isna().sum().sum()):,}")


File: F:\Project\IDS-ML-Thesis\data\processed\identified_flows.parquet
Rows: 2,827,876
Columns: 74
Labels: 15
Missing values: 0


## 2. Label distribution

In [8]:
label_counts = df["Label"].value_counts()
display(label_counts.to_frame("rows"))
print("Total labeled rows:", f"{int(label_counts.sum()):,}")


,rows
Label,
BENIGN,2271320
DoS Hulk,230124
PortScan,158804
DDoS,128025
DoS GoldenEye,10293
FTP-Patator,7935
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


Total labeled rows: 2,827,876


## 3. Check identifier completeness

In [9]:
IDENTIFIER_COLUMNS = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
]

for column in IDENTIFIER_COLUMNS:
    if column not in df.columns:
        print(f"{column}: MISSING COLUMN")
    else:
        missing = int(df[column].isna().sum())
        blank = int(df[column].astype("string").str.strip().eq("").sum())
        print(f"{column}: missing={missing:,}, blank={blank:,}")


Flow ID: missing=0, blank=0
Source IP: missing=0, blank=0
Source Port: missing=0, blank=0
Destination IP: missing=0, blank=0
Destination Port: missing=0, blank=0
Protocol: missing=0, blank=0
Timestamp: missing=0, blank=0


## 4. Exact duplicate-row diagnostic

This count uses every column in the processed dataset, including identifiers. It is a diagnostic only; the experimental duplicate policy must be defined using the selected model features and label.


In [10]:
print("Exact duplicate rows:", f"{int(df.duplicated().sum()):,}")


Exact duplicate rows: 199


## 5. Five-tuple diagnostic

A five-tuple consists of source IP, source port, destination IP, destination port, and protocol. A tuple may recur at different times, so it is not assumed to identify one unique flow forever.


In [11]:
GROUP_COLUMNS = [
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
]

group_sizes = df.groupby(
    GROUP_COLUMNS, dropna=False, sort=False
).size()

labels_per_tuple = df.groupby(
    GROUP_COLUMNS, dropna=False, sort=False
)["Label"].nunique()

print("Valid rows:", f"{len(df):,}")
print("Unique five-tuples:", f"{len(group_sizes):,}")
print("Repeated five-tuple groups:", f"{int((group_sizes > 1).sum()):,}")
print(
    "Rows in repeated groups:",
    f"{int(group_sizes[group_sizes > 1].sum()):,}"
)
print(
    "Five-tuples associated with multiple labels:",
    f"{int((labels_per_tuple > 1).sum()):,}"
)


Valid rows: 2,827,876
Unique five-tuples: 1,456,266
Repeated five-tuple groups: 472,506
Rows in repeated groups: 1,844,116
Five-tuples associated with multiple labels: 13,069


## 6. Prepare labels and model features

Identifier columns are retained in `df` for grouping and analysis, but are excluded from `X` so they are not accidentally supplied as predictive features.


In [12]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))
import data

X, y, groups = data.load(path=DATA_PATH)
X_without_port, y_without_port, groups_without_port = data.load(
    path=DATA_PATH, include_destination_port=False
)

print("Default feature matrix shape:", X.shape)
print("Target length:", len(y))
print("Group count:", len(groups))
print("Features without Destination Port:", X_without_port.shape[1])
print("Labels unchanged:", y.equals(y_without_port))
print("Groups unchanged:", groups.equals(groups_without_port))
print("Non-numeric features:", X.select_dtypes(exclude="number").columns.tolist())


Default feature matrix shape: (2827876, 67)
Target length: 2827876
Group count: 2827876
Features without Destination Port: 66
Labels unchanged: True
Groups unchanged: True
Non-numeric features: []


## Audit notes

- The preprocessing script writes a separate identified-flow Parquet file.
- The legacy processed dataset is not overwritten.
- Exact duplicate handling and the final split strategy remain explicit experimental decisions.
- Five-tuple grouping reduces some overlap risks but does not guarantee a completely leakage-free evaluation.
